In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

GLOBAL_DIR = (
    PROJECT_ROOT
    / "results"
    / "global_inventory"
)

session_df = pd.read_csv(
    GLOBAL_DIR / "global_session_summary.csv"
)

region_df = pd.read_csv(
    GLOBAL_DIR / "global_region_inventory.csv"
)

trial_df = pd.read_csv(
    GLOBAL_DIR / "global_trial_condition_inventory.csv"
)

errors_df = pd.read_csv(
    GLOBAL_DIR / "global_inventory_errors.csv"
)

print("Sessions:", session_df.shape)
print("Regions:", region_df.shape)
print("Trial conditions:", trial_df.shape)
print("Errors:", errors_df.shape)

Sessions: (459, 22)
Regions: (8231, 9)
Trial conditions: (24472, 12)
Errors: (3, 7)


In [2]:
print(
    "Contrasts:",
    sorted(
        trial_df[
            "gabor_stimulus_contrast"
        ]
        .dropna()
        .unique()
    )
)

print("\nStimulus sides:")
display(
    trial_df[
        "gabor_stimulus_side"
    ].value_counts(dropna=False)
)

print("\nProbability-left values:")
display(
    trial_df[
        "probability_left"
    ].value_counts(dropna=False)
)

Contrasts: [np.float64(0.0), np.float64(6.25), np.float64(12.5), np.float64(25.0), np.float64(100.0)]

Stimulus sides:


gabor_stimulus_side
left     12295
right    12177
Name: count, dtype: int64


Probability-left values:


probability_left
0.8    8625
0.2    8404
0.5    7443
Name: count, dtype: int64

In [3]:
TARGET_REGION = "Primary visual area"

MIN_GOOD_UNITS = 15
MIN_TOTAL_VALID_TRIALS = 200
MIN_TRIALS_PER_CONDITION = 10

HIGH_CONTRASTS = {100.0}
LOW_CONTRASTS = {6.25, 12.5}

KEYS = [
    "subject",
    "session",
    "asset_path",
]

# 临时按 parent region 合并不同 layer
region_session_df = (
    region_df
    .groupby(
        KEYS + ["region_parent"],
        as_index=False,
        dropna=False,
    )
    .agg(
        n_all_units=("n_all_units", "sum"),
        n_good_units=("n_good_units", "sum"),
        n_region_subdivisions=("region_raw", "nunique"),
    )
)

# 找出包含目标脑区的 session
target_region_session_df = (
    region_session_df.loc[
        region_session_df["region_parent"]
        .eq(TARGET_REGION)
    ]
    .copy()
)

display(
    target_region_session_df
    .sort_values(
        "n_good_units",
        ascending=False,
    )
    .head(30)
)


,subject,session,asset_path,region_parent,n_all_units,n_good_units,n_region_subdivisions
173,CSH-ZAD-022,41431f53-69fd-4e3b-80ce-ea62e03bf9c7,sub-CSH-ZAD-022/sub-CSH-ZAD-022_ses-41431f53-6...,Primary visual area,829,63,6
352,CSH-ZAD-025,6274dda8-3a59-4aa1-95f8-a8a549c46a26,sub-CSH-ZAD-025/sub-CSH-ZAD-025_ses-6274dda8-3...,Primary visual area,300,62,4
5810,ZFM-01577,09b2c4d1-058d-4c84-9fd4-97530f85baf6,sub-ZFM-01577/sub-ZFM-01577_ses-09b2c4d1-058d-...,Primary visual area,264,51,5
938,CSHL052,85dc2ebd-8aaf-46b0-9284-a197aee8b16f,sub-CSHL052/sub-CSHL052_ses-85dc2ebd-8aaf-46b0...,Primary visual area,222,49,4
4617,PL050,2584ce3c-db10-4076-89cb-5d313138dd38,sub-PL050/sub-PL050_ses-2584ce3c-db10-4076-89c...,Primary visual area,296,44,4
3016,NR-0021,e0928e11-2b86-4387-a203-80c77fab5d52,sub-NR-0021/sub-NR-0021_ses-e0928e11-2b86-4387...,Primary visual area,276,40,6
2727,MFD-05,ca4ecb4c-4b60-4723-9b9e-2c54a6290a53,sub-MFD-05/sub-MFD-05_ses-ca4ecb4c-4b60-4723-9...,Primary visual area,232,40,6
594,CSH-ZAD-029,e349a2e7-50a3-47ca-bc45-20d1899854ec,sub-CSH-ZAD-029/sub-CSH-ZAD-029_ses-e349a2e7-5...,Primary visual area,542,40,5
3200,NR-0028,5c0c560e-9e1f-45e9-b66e-e4ee7855be84,sub-NR-0028/sub-NR-0028_ses-5c0c560e-9e1f-45e9...,Primary visual area,547,36,6
434,CSH-ZAD-026,5d01d14e-aced-4465-8f8e-9a1c674f62ec,sub-CSH-ZAD-026/sub-CSH-ZAD-026_ses-5d01d14e-a...,Primary visual area,175,34,6


In [7]:
#标签，长表
def classify_contrast(value):
    if value in HIGH_CONTRASTS:
        return "high"

    if value in LOW_CONTRASTS:
        return "low"

    return pd.NA


trial_screen_df = trial_df.copy()

trial_screen_df["contrast_group"] = (
    trial_screen_df[
        "gabor_stimulus_contrast"
    ]
    .apply(classify_contrast)
    .astype("string")
)

trial_screen_df = (
    trial_screen_df
    .dropna(
        subset=[
            "contrast_group",
            "gabor_stimulus_side",
        ]
    )
    .copy()
)

trial_condition_counts = (
    trial_screen_df
    .groupby(
        KEYS
        + [
            "contrast_group",
            "gabor_stimulus_side",
        ],
        as_index=False,
    )
    .agg(
        n_valid_trials=(
            "n_valid_trials",
            "sum",
        )
    )
)

display(trial_condition_counts.head(12))

,subject,session,asset_path,contrast_group,gabor_stimulus_side,n_valid_trials
0,CSH-ZAD-001,3e7ae7c0-fe8b-487c-9354-036236fa1010,sub-CSH-ZAD-001/sub-CSH-ZAD-001_ses-3e7ae7c0-f...,high,left,39
1,CSH-ZAD-001,3e7ae7c0-fe8b-487c-9354-036236fa1010,sub-CSH-ZAD-001/sub-CSH-ZAD-001_ses-3e7ae7c0-f...,high,right,57
2,CSH-ZAD-001,3e7ae7c0-fe8b-487c-9354-036236fa1010,sub-CSH-ZAD-001/sub-CSH-ZAD-001_ses-3e7ae7c0-f...,low,left,97
3,CSH-ZAD-001,3e7ae7c0-fe8b-487c-9354-036236fa1010,sub-CSH-ZAD-001/sub-CSH-ZAD-001_ses-3e7ae7c0-f...,low,right,85
4,CSH-ZAD-011,03cf52f6-fba6-4743-a42e-dd1ac3072343,sub-CSH-ZAD-011/sub-CSH-ZAD-011_ses-03cf52f6-f...,high,left,48
5,CSH-ZAD-011,03cf52f6-fba6-4743-a42e-dd1ac3072343,sub-CSH-ZAD-011/sub-CSH-ZAD-011_ses-03cf52f6-f...,high,right,31
6,CSH-ZAD-011,03cf52f6-fba6-4743-a42e-dd1ac3072343,sub-CSH-ZAD-011/sub-CSH-ZAD-011_ses-03cf52f6-f...,low,left,71
7,CSH-ZAD-011,03cf52f6-fba6-4743-a42e-dd1ac3072343,sub-CSH-ZAD-011/sub-CSH-ZAD-011_ses-03cf52f6-f...,low,right,60
8,CSH-ZAD-011,5b44c40f-80f4-44fb-abfb-c7f19e27a6ca,sub-CSH-ZAD-011/sub-CSH-ZAD-011_ses-5b44c40f-8...,high,left,67
9,CSH-ZAD-011,5b44c40f-80f4-44fb-abfb-c7f19e27a6ca,sub-CSH-ZAD-011/sub-CSH-ZAD-011_ses-5b44c40f-8...,high,right,57


In [8]:
#长表转宽表
trial_wide_df = (
    trial_condition_counts
    .pivot_table(
        index=KEYS,
        columns=[
            "contrast_group",
            "gabor_stimulus_side",
        ],
        values="n_valid_trials",
        fill_value=0,
    )
)

trial_wide_df.columns = [
    f"n_{contrast_group}_{stimulus_side}"
    for contrast_group, stimulus_side
    in trial_wide_df.columns
]

trial_wide_df = (
    trial_wide_df
    .reset_index()
)

display(trial_wide_df.head())

,subject,session,asset_path,n_high_left,n_high_right,n_low_left,n_low_right
0,CSH-ZAD-001,3e7ae7c0-fe8b-487c-9354-036236fa1010,sub-CSH-ZAD-001/sub-CSH-ZAD-001_ses-3e7ae7c0-f...,39.0,57.0,97.0,85.0
1,CSH-ZAD-011,03cf52f6-fba6-4743-a42e-dd1ac3072343,sub-CSH-ZAD-011/sub-CSH-ZAD-011_ses-03cf52f6-f...,48.0,31.0,71.0,60.0
2,CSH-ZAD-011,5b44c40f-80f4-44fb-abfb-c7f19e27a6ca,sub-CSH-ZAD-011/sub-CSH-ZAD-011_ses-5b44c40f-8...,67.0,57.0,107.0,78.0
3,CSH-ZAD-011,da188f2c-553c-4e04-879b-c9ea2d1b9a93,sub-CSH-ZAD-011/sub-CSH-ZAD-011_ses-da188f2c-5...,94.0,68.0,156.0,130.0
4,CSH-ZAD-019,49e0ab27-827a-4c91-bcaa-97eea27a1b8d,sub-CSH-ZAD-019/sub-CSH-ZAD-019_ses-49e0ab27-8...,82.0,50.0,127.0,113.0


In [10]:
# 1. 定义必须存在的四个 trial-condition 计数列
REQUIRED_COUNT_COLUMNS = [
    "n_high_left",
    "n_high_right",
    "n_low_left",
    "n_low_right",
]

# 如果整个数据集中某个组合完全不存在，则补一列 0。
for column in REQUIRED_COUNT_COLUMNS:
    if column not in trial_wide_df.columns:
        trial_wide_df[column] = 0


# 2. 选择 session-level summary 中需要的列
session_columns = KEYS + [
    "n_units",
    "n_good_units",
    "n_trials",
    "n_valid_trials",
    "valid_trial_fraction",
    "median_rt_valid_s",
    "accuracy_valid",
]
# 3. 合并三类信息
# target_region_df: 目标脑区的 unit 数量
# session_df: 整个 session 的 unit 和行为 QC
# trial_wide_df: high/low × left/right 的有效 trial 数

candidate_df = (
    target_region_session_df
    .merge(
        session_df[session_columns],
        on=KEYS,
        how="left",
        validate="one_to_one",
        suffixes=(
            "_target_region",
            "_whole_session",
        ),
    )
    .merge(
        trial_wide_df,
        on=KEYS,
        how="left",
        validate="one_to_one",
    )
)


candidate_df[REQUIRED_COUNT_COLUMNS] = (
    candidate_df[REQUIRED_COUNT_COLUMNS]
    .fillna(0)
    .astype(int)
)


# 5. 创建筛选条件
# 条件 1：
# 目标脑区中至少有 MIN_GOOD_UNITS 个 good units
candidate_df["passes_region"] = (
    candidate_df[
        "n_good_units_target_region"
    ]
    .ge(MIN_GOOD_UNITS)
)


# 条件 2：
# 整个 session 至少有 MIN_TOTAL_VALID_TRIALS 个 valid trials
candidate_df["passes_total_trials"] = (
    candidate_df[
        "n_valid_trials"
    ]
    .fillna(0)
    .ge(MIN_TOTAL_VALID_TRIALS)
)


# 四个核心条件中的最小 trial 数
candidate_df[
    "min_trials_across_conditions"
] = (
    candidate_df[
        REQUIRED_COUNT_COLUMNS
    ]
    .min(axis=1)
)


# 条件 3：
# high-left、high-right、low-left、low-right
# 每一种条件都至少有 MIN_TRIALS_PER_CONDITION 个 trials
candidate_df["passes_condition_trials"] = (
    candidate_df[
        "min_trials_across_conditions"
    ]
    .ge(MIN_TRIALS_PER_CONDITION)
)


# 当前 high/low × left/right 四类中共有多少 valid trials
candidate_df["n_screen_trials"] = (
    candidate_df[
        REQUIRED_COUNT_COLUMNS
    ]
    .sum(axis=1)
)


# 最终筛选：
# 三个条件必须同时通过
candidate_df["passes_screen"] = (
    candidate_df[
        [
            "passes_region",
            "passes_total_trials",
            "passes_condition_trials",
        ]
    ]
    .all(axis=1)
)

# 6. candidate session ranking

candidate_df = (
    candidate_df
    .sort_values(
        [
            "passes_screen",
            "min_trials_across_conditions",
            "n_good_units_target_region",
            "n_valid_trials",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)


# 7. 查看排名最高的候选 session

display(
    candidate_df[
        [
            "subject",
            "session",
            "n_good_units_target_region",
            "n_good_units_whole_session",
            "n_valid_trials",
            "valid_trial_fraction",
            "accuracy_valid",
            "n_high_left",
            "n_high_right",
            "n_low_left",
            "n_low_right",
            "n_screen_trials",
            "min_trials_across_conditions",
            "passes_region",
            "passes_total_trials",
            "passes_condition_trials",
            "passes_screen",
        ]
    ]
    .head(30)
)

,subject,session,n_good_units_target_region,n_good_units_whole_session,n_valid_trials,valid_trial_fraction,accuracy_valid,n_high_left,n_high_right,n_low_left,n_low_right,n_screen_trials,min_trials_across_conditions,passes_region,passes_total_trials,passes_condition_trials,passes_screen
0,CSH-ZAD-029,e349a2e7-50a3-47ca-bc45-20d1899854ec,40,198,883,0.863992,0.906002,109,106,211,177,603,106,True,True,True,True
1,NR-0028,5c0c560e-9e1f-45e9-b66e-e4ee7855be84,36,277,843,0.766364,0.836299,115,98,201,149,563,98,True,True,True,True
2,CSH-ZAD-025,90c61c38-b9fd-4cc3-9795-29160d2f8e55,20,116,656,0.815920,0.835366,62,77,135,159,433,62,True,True,True,True
3,CSH-ZAD-025,6274dda8-3a59-4aa1-95f8-a8a549c46a26,62,376,590,0.755442,0.903390,97,61,164,80,402,61,True,True,True,True
4,ibl-witten-29,58b1e920-cfc8-467e-b28b-7654a55d0977,24,44,565,0.776099,0.907965,63,52,122,135,372,52,True,True,True,True
5,PL050,0cf6d255-8f2f-463e-84fb-c54bacb79f51,24,110,444,0.632479,0.864865,49,50,98,105,302,49,True,True,True,True
6,ZFM-01577,09b2c4d1-058d-4c84-9fd4-97530f85baf6,51,315,424,0.709030,0.830189,62,46,91,81,280,46,True,True,True,True
7,UCLA015,0c828385-6dd6-4842-a702-c5075f5f5e81,24,133,472,0.773770,0.822034,74,46,114,87,321,46,True,True,True,True
8,CSHL052,85dc2ebd-8aaf-46b0-9284-a197aee8b16f,49,146,410,0.532468,0.853659,44,46,93,84,267,44,True,True,True,True
9,NR-0021,e0928e11-2b86-4387-a203-80c77fab5d52,40,92,322,0.679325,0.891304,38,41,62,63,204,38,True,True,True,True


In [11]:
#保存候选表
OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "candidate_sessions"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

candidate_path = (
    OUTPUT_DIR
    / "VISp_candidate_sessions.csv"
)

candidate_df.to_csv(
    candidate_path,
    index=False,
)

print("Saved:", candidate_path)

Saved: D:\Ephys_analysis\ibl-population-dynamics\results\candidate_sessions\VISp_candidate_sessions.csv


In [12]:
# 建立最终通过筛选的 session 表

eligible_df = (
    candidate_df.loc[
        candidate_df["passes_screen"]
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Sessions containing target region:",
    len(candidate_df),
)

print(
    "Eligible sessions:",
    len(eligible_df),
)

print(
    "Unique eligible subjects:",
    eligible_df["subject"].nunique(),
)


# 每只动物贡献多少个 eligible sessions
sessions_per_subject = (
    eligible_df
    .groupby(
        "subject",
        as_index=False,
    )
    .agg(
        n_eligible_sessions=(
            "session",
            "nunique",
        ),
        median_target_good_units=(
            "n_good_units_target_region",
            "median",
        ),
        median_min_condition_trials=(
            "min_trials_across_conditions",
            "median",
        ),
    )
    .sort_values(
        [
            "n_eligible_sessions",
            "median_target_good_units",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(sessions_per_subject)

Sessions containing target region: 60
Eligible sessions: 21
Unique eligible subjects: 19


,subject,n_eligible_sessions,median_target_good_units,median_min_condition_trials
0,CSH-ZAD-025,2,41.0,61.5
1,PL050,2,34.0,39.0
2,CSH-ZAD-022,1,63.0,31.0
3,ZFM-01577,1,51.0,46.0
4,CSHL052,1,49.0,44.0
5,CSH-ZAD-029,1,40.0,106.0
6,MFD-05,1,40.0,24.0
7,NR-0021,1,40.0,38.0
8,NR-0028,1,36.0,98.0
9,CSH-ZAD-026,1,34.0,19.0


In [13]:
display(
    eligible_df[
        [
            "n_good_units_target_region",
            "n_valid_trials",
            "valid_trial_fraction",
            "accuracy_valid",
            "n_high_left",
            "n_high_right",
            "n_low_left",
            "n_low_right",
            "min_trials_across_conditions",
        ]
    ]
    .describe()
    .T
)

,count,mean,std,min,25%,50%,75%,max
n_good_units_target_region,21.0,34.142857,13.940178,17.000000,24.000000,33.000000,40.000000,63.000000
n_valid_trials,21.0,430.190476,185.142275,207.000000,315.000000,393.000000,472.000000,883.000000
valid_trial_fraction,21.0,0.633535,0.159163,0.279730,0.532468,0.689162,0.755442,0.863992
accuracy_valid,21.0,0.848220,0.052395,0.750685,0.822034,0.847619,0.891304,0.952522
n_high_left,21.0,54.809524,26.183237,23.000000,38.000000,49.000000,62.000000,115.000000
n_high_right,21.0,47.619048,22.608574,19.000000,32.000000,43.000000,52.000000,106.000000
n_low_left,21.0,98.761905,46.476774,43.000000,68.000000,91.000000,114.000000,211.000000
n_low_right,21.0,86.095238,40.122194,28.000000,61.000000,80.000000,105.000000,177.000000
min_trials_across_conditions,21.0,43.571429,22.850758,19.000000,29.000000,38.000000,49.000000,106.000000


In [15]:
OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "candidate_sessions"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

all_candidate_path = (
    OUTPUT_DIR
    / "VISp_all_candidate_sessions.csv"
)

eligible_path = (
    OUTPUT_DIR
    / "VISp_eligible_sessions.csv"
)

candidate_df.to_csv(
    all_candidate_path,
    index=False,
)

eligible_df.to_csv(
    eligible_path,
    index=False,
)

print("Saved:", all_candidate_path)
print("Saved:", eligible_path)

Saved: D:\Ephys_analysis\ibl-population-dynamics\results\candidate_sessions\VISp_all_candidate_sessions.csv
Saved: D:\Ephys_analysis\ibl-population-dynamics\results\candidate_sessions\VISp_eligible_sessions.csv


In [16]:
print("session_df:")
print(session_df.columns.tolist())

print("\nregion_df:")
print(region_df.columns.tolist())

print("\ntrial_df:")
print(trial_df.columns.tolist())

session_df:
['subject', 'session', 'asset_path', 'n_units', 'n_good_units', 'n_units_with_region', 'n_probes', 'probe_names', 'n_trials', 'n_valid_trials', 'valid_trial_fraction', 'n_missing_required_events', 'n_rt_out_of_range', 'n_contrasts', 'n_stimulus_sides', 'n_probability_left_values', 'n_block_types', 'median_rt_valid_s', 'accuracy_valid', 'dandiset_id', 'dandiset_version', 'processed_at_utc']

region_df:
['subject', 'session', 'asset_path', 'region_raw', 'region_parent', 'n_all_units', 'n_good_units', 'n_probes', 'median_firing_rate_hz']

trial_df:
['subject', 'session', 'asset_path', 'gabor_stimulus_contrast', 'gabor_stimulus_side', 'probability_left', 'block_type', 'mouse_wheel_choice', 'is_mouse_rewarded', 'n_trials', 'n_valid_trials', 'median_rt_valid_s', 'contrast_group']
